In [7]:
import torch
from torch import nn
import torch.functional as F 
import math

## Batch and Layer Normalizations

In [5]:
def batch_norm(X, gamma, beta, moving_mean, moving_var, eps, momentum):
  if not torch.is_grad_enabled(): # In prediction mode
    X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps)
  else:
    assert len(X.shape) in (2, 4)
    if len(X.shape) == 2:
      mean = X.mean(dim=0)
      var = ((X - mean)**2).mean(dim=0)
    else:
      mean = X.mean(dim=(0, 2, 3), keepdim=True)
      var = ((X - mean)**2).mean(dim=(0, 2, 3), keepdim=True)
    X_hat = (X - mean) / torch.sqrt(var + eps)
    moving_mean = momentum * moving_mean + (1.0 - momentum) * mean
    moving_var = momentum * moving_var + (1.0 - momentum) * var
  
  Y = gamma * X_hat + beta
  return Y, moving_mean, moving_var

## Residual Connections

In [6]:
class ResidualBlock(nn.Module):
  def __init__(self, input_channels, num_channels):
    super().__init__()
    self.conv1 = nn.Conv2d(
      input_channels, num_channels, kernel_size=3, padding=1
    )
    self.conv2 = nn.Conv2d(
      num_channels, num_channels, kernel_size=3, padding=1
    )
    self.bn1 = nn.BatchNorm2d(num_channels)
    self.bn2 = nn.BatchNorm2d(num_channels)
    
  def forward(self, X):
    Y = F.relu(self.bn1(self.conv1(X)))
    Y = self.bn2(self.conv2(Y))
    return F.relu(Y + X)

### Attention

In [8]:
# Shape of `queries`: (batch_size, #queries, d)
# Shape of `keys`: (batch_size, #keys, d)
# Shape of `values`: (batch_size, #values, d)

def dot_product_attention(queries, keys, values):
  d = queries.shape[-1]
  scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
  return torch.bmm(F.softmax(scores, dim=-1), values)

In [ ]:
def multi_head_attention(queries, keys, values, n_head):
  outputs = []
  for i in range(n_head):
    # W_q, W_k, W_v are linear layers\
    outputs.append(dot_product_attention(
      W_q(queries), W_k(keys), W_v(values)
    ))
  return W_o(torch.cat(outputs, dim=-1)) # W_o is a linear layer 

ffn = nn.Sequential(nn.Linear(num_inputs, num_hiddens), nn.ReLU(), 
                    nn.Linear(num_hiddens, num_outputs))

# shape of input `X`: (batch_size, seq_len, d)
def transformer_block(X):
  Y = nn.LayerNorm()(multi_head_attention(X, X, X) + X)
  return nn.LayerNorm()(ffn(Y) + Y)